# 格子ベース暗号：CRYSTALS-Kyber & Dilithium

## NISTが選定したPQC標準 (2024年8月 FIPS化)

```
FIPS 203: CRYSTALS-Kyber   → 鍵カプセル化 (KEM)
FIPS 204: CRYSTALS-Dilithium → デジタル署名
FIPS 205: SPHINCS+          → デジタル署名 (ハッシュベース)
```

### 格子ベース暗号の安全性根拠

**LWE問題 (Learning With Errors)**:
- $b = As + e \pmod{q}$ の形の連立方程式から $s$ を求めることは困難
- $e$: 小さな「ノイズ」ベクトル
- 量子コンピュータでも効率的な解法が存在しない

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import norm
import secrets
import hashlib
import struct

print('格子暗号ライブラリの読み込み完了')

## 1. LWE問題の可視化

In [ ]:
def visualize_lwe_problem():
    """LWE問題の直感的な理解"""
    np.random.seed(42)
    n = 2   # 秘密ベクトルの次元（可視化用に2次元）
    q = 97  # 素数モジュラス
    m = 20  # サンプル数

    # 秘密ベクトル s（これを守る）
    s = np.array([23, 51])

    # 公開行列 A（ランダム）
    A = np.random.randint(0, q, size=(m, n))

    # ノイズなしの場合
    b_exact = (A @ s) % q

    # ノイズありの場合（LWEのノイズ）
    noise_std = 2.0
    e = np.round(np.random.normal(0, noise_std, m)).astype(int)
    b_noisy = (A @ s + e) % q

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # LWEサンプルの可視化
    axes[0].scatter(A[:, 0], b_exact, c='blue', alpha=0.7, label='ノイズなし (As mod q)', s=60)
    axes[0].scatter(A[:, 0], b_noisy, c='red', alpha=0.7, label='LWEサンプル (As+e mod q)', marker='x', s=80)
    axes[0].set_xlabel('A の第1列', fontsize=11)
    axes[0].set_ylabel('b の値', fontsize=11)
    axes[0].set_title(f'LWE問題の可視化\ns = {s}, q = {q}\n「どの s がノイズ付きデータを生成したか？」', fontsize=12)
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)

    # ノイズ分布
    x = np.arange(-8, 9)
    from scipy.stats import norm as sp_norm
    axes[1].bar(x, [np.sum(e == i) for i in x], color='steelblue', alpha=0.7, label='実際のノイズ')
    x_cont = np.linspace(-8, 8, 100)
    axes[1].plot(x_cont, m * sp_norm.pdf(x_cont, 0, noise_std), 'r-', linewidth=2, label=f'正規分布 σ={noise_std}')
    axes[1].set_xlabel('ノイズ値', fontsize=11)
    axes[1].set_ylabel('出現回数', fontsize=11)
    axes[1].set_title('LWEノイズ分布\n（小さなノイズが安全性の鍵）', fontsize=12)
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('lwe_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'秘密ベクトル s = {s}')
    print(f'公開行列 A の形状: {A.shape}')
    print(f'ノイズ e の範囲: [{e.min()}, {e.max()}]')
    print(f'攻撃者には (A, b_noisy) のみが公開される')

visualize_lwe_problem()

## 2. CRYSTALS-Kyber の簡略実装

### Module-LWEベースの鍵カプセル化メカニズム (KEM)

```
KeyGen: (pk, sk) ← KeyGen()
Encaps: (ct, K)  ← Encaps(pk)    # 共通鍵 K を生成し暗号化
Decaps: K        ← Decaps(sk, ct) # 共通鍵 K を復元
```

In [ ]:
class SimpleKyber:
    """
    CRYSTALS-Kyberの教育用簡略実装
    注意: 実際のKyberはNTTを使った多項式環上で動作
    これは概念理解のための簡略版です
    """
    
    def __init__(self, n=256, q=3329, k=2, eta1=3, eta2=2):
        """
        パラメータ:
        n: 多項式次数 (Kyber: 256)
        q: モジュラス (Kyber: 3329)
        k: モジュール次元 (Kyber512: 2, Kyber768: 3, Kyber1024: 4)
        eta1, eta2: ノイズ分布パラメータ
        """
        self.n = n
        self.q = q
        self.k = k
        self.eta1 = eta1
        self.eta2 = eta2
        # 実際のKyberで使う圧縮パラメータ
        self.du = 10  # Kyber512の場合
        self.dv = 4
    
    def _centered_binomial(self, eta, size):
        """中心二項分布からのサンプリング（Kyberのノイズ）"""
        # eta個のビットの差：(sum of eta bits) - (sum of eta bits)
        a = np.random.randint(0, 2, size=(*size, eta)).sum(axis=-1)
        b = np.random.randint(0, 2, size=(*size, eta)).sum(axis=-1)
        return (a - b) % self.q
    
    def _poly_mul_mod(self, f, g):
        """多項式乗算 mod (X^n + 1) mod q"""
        # 実際のKyberはNTT(数論変換)を使う
        result = np.zeros(self.n, dtype=np.int64)
        for i in range(self.n):
            for j in range(self.n):
                idx = (i + j) % self.n
                sign = 1 if (i + j) < self.n else -1
                result[idx] = (result[idx] + sign * f[i] * g[j]) % self.q
        return result
    
    def _matrix_poly_mul(self, A, v):
        """行列-ベクトル多項式乗算"""
        result = []
        for i in range(self.k):
            acc = np.zeros(self.n, dtype=np.int64)
            for j in range(self.k):
                acc = (acc + self._poly_mul_mod(A[i][j], v[j])) % self.q
            result.append(acc)
        return result
    
    def keygen(self):
        """鍵生成"""
        # 公開行列 A の生成（ランダムシードから展開）
        rho = secrets.token_bytes(32)  # シード
        A = [[np.random.randint(0, self.q, self.n) 
              for _ in range(self.k)] 
             for _ in range(self.k)]
        
        # 秘密鍵 s と誤差 e の生成
        s = [self._centered_binomial(self.eta1, (self.n,)) for _ in range(self.k)]
        e = [self._centered_binomial(self.eta1, (self.n,)) for _ in range(self.k)]
        
        # 公開鍵 t = As + e
        As = self._matrix_poly_mul(A, s)
        t = [(As[i] + e[i]) % self.q for i in range(self.k)]
        
        pk = (A, t)
        sk = s
        return pk, sk
    
    def encaps_simple(self, pk, message_bit):
        """簡略カプセル化（1ビットのメッセージ）"""
        A, t = pk
        
        # エフェメラルなランダム値
        r = [self._centered_binomial(self.eta1, (self.n,)) for _ in range(self.k)]
        e1 = [self._centered_binomial(self.eta2, (self.n,)) for _ in range(self.k)]
        e2 = self._centered_binomial(self.eta2, (self.n,))
        
        # u = A^T r + e1
        AT = [[A[j][i] for j in range(self.k)] for i in range(self.k)]
        ATr = self._matrix_poly_mul(AT, r)
        u = [(ATr[i] + e1[i]) % self.q for i in range(self.k)]
        
        # v = t^T r + e2 + round(q/2) * m
        tr_dot = np.zeros(self.n, dtype=np.int64)
        for i in range(self.k):
            tr_dot = (tr_dot + self._poly_mul_mod(t[i], r[i])) % self.q
        
        m_encoded = np.full(self.n, message_bit * (self.q // 2), dtype=np.int64)
        v = (tr_dot + e2 + m_encoded) % self.q
        
        return (u, v)
    
    def decaps_simple(self, sk, ct):
        """簡略カプセル化解除"""
        u, v = ct
        s = sk
        
        # s^T u の計算
        su_dot = np.zeros(self.n, dtype=np.int64)
        for i in range(self.k):
            su_dot = (su_dot + self._poly_mul_mod(s[i], u[i])) % self.q
        
        # m' = v - s^T u (近似デコード)
        m_prime = (v - su_dot) % self.q
        
        # 量子化: q/4 に近いなら 1、0に近いなら 0
        threshold = self.q // 2
        quarter = self.q // 4
        decoded = 1 if abs(m_prime[0] - threshold) < quarter else 0
        
        return decoded


# Kyberの動作確認（簡略版、小さなパラメータで）
print('CRYSTALS-Kyber 簡略実装テスト')
print('=' * 50)
print('注意: 概念理解用の簡略実装です')
print()

# 実際のKyberパラメータの表示
kyber_params = {
    'Kyber512 (ML-KEM-512)':  {'n': 256, 'q': 3329, 'k': 2, '安全レベル': 'AES-128相当', 'NIST Level': 1},
    'Kyber768 (ML-KEM-768)':  {'n': 256, 'q': 3329, 'k': 3, '安全レベル': 'AES-192相当', 'NIST Level': 3},
    'Kyber1024 (ML-KEM-1024)': {'n': 256, 'q': 3329, 'k': 4, '安全レベル': 'AES-256相当', 'NIST Level': 5},
}

print('Kyberパラメータセット:')
print(f'{"変種":<25} {"n":<6} {"q":<6} {"k":<4} {"安全レベル":<15} {"NISTレベル"}')
print('-' * 65)
for name, params in kyber_params.items():
    print(f'{name:<25} {params["n"]:<6} {params["q"]:<6} {params["k"]:<4} {params["安全レベル"]:<15} Level {params["NIST Level"]}')

## 3. CRYSTALS-Dilithium：デジタル署名

### 署名の仕組み（Module-LWE + Module-SIS）

In [ ]:
class SimpleDilithium:
    """
    CRYSTALS-Dilithiumの教育用概念実装
    Fiat-Shamir変換による署名スキーム
    """
    
    def __init__(self, q=8380417, n=256, k=4, l=4, eta=2, beta=78, omega=80):
        """
        Dilithium3パラメータ (FIPS 204)
        q: モジュラス (2^23 - 2^13 + 1、NTT友好的な素数)
        n: 多項式次数
        k, l: 行列次元
        eta: 秘密鍵係数の範囲
        """
        self.q = q
        self.n = n
        self.k = k
        self.l = l
        self.eta = eta
        self.beta = beta
        self.omega = omega
    
    def _hash_to_challenge(self, mu, w1):
        """チャレンジ多項式の生成（ランダムオラクル）"""
        # 実際は SHAKE256 を使用
        data = mu + str(w1).encode()
        h = hashlib.shake_256(data).digest(32)
        # tau個の非ゼロ係数を持つ多項式
        tau = 49  # Dilithium3の場合
        challenge = np.zeros(self.n, dtype=np.int64)
        rng = np.random.RandomState(int.from_bytes(h[:4], 'big'))
        positions = rng.choice(self.n, tau, replace=False)
        signs = rng.randint(0, 2, tau) * 2 - 1
        for pos, sign in zip(positions, signs):
            challenge[pos] = sign
        return challenge
    
    def keygen(self):
        """Dilithium鍵生成"""
        # 公開行列 A ∈ R_q^{k×l}
        seed = secrets.token_bytes(32)
        np.random.seed(int.from_bytes(seed[:4], 'big'))
        
        A = np.random.randint(0, self.q, (self.k, self.l, self.n))
        
        # 秘密鍵 s1, s2 (小さな係数)
        s1 = np.random.randint(-self.eta, self.eta + 1, (self.l, self.n))
        s2 = np.random.randint(-self.eta, self.eta + 1, (self.k, self.n))
        
        # 公開鍵 t = As1 + s2 mod q
        As1 = np.zeros((self.k, self.n), dtype=np.int64)
        for i in range(self.k):
            for j in range(self.l):
                # 簡略: 多項式乗算を係数積で近似
                As1[i] = (As1[i] + A[i, j] * s1[j, 0]) % self.q
        
        t = (As1 + s2) % self.q
        
        pk = (A, t)
        sk = (A, t, s1, s2)
        return pk, sk
    
    def sign_concept(self, sk, message):
        """署名の概念的な流れ（簡略版）"""
        A, t, s1, s2 = sk
        mu = hashlib.sha3_256(message).digest()
        
        # Reject sampling ループ
        for attempt in range(100):
            # 1. マスク y を生成
            gamma1 = 2**19  # Dilithium3
            y = np.random.randint(-gamma1, gamma1, (self.l, self.n))
            
            # 2. w = Ay を計算
            Ay = np.zeros((self.k, self.n), dtype=np.int64)
            for i in range(self.k):
                for j in range(self.l):
                    Ay[i] = (Ay[i] + A[i, j] * y[j, 0]) % self.q
            
            # 3. w1 = HighBits(Ay)
            gamma2 = (self.q - 1) // 88
            w1 = (Ay // (2 * gamma2)) % self.q
            
            # 4. チャレンジ c を生成
            c = self._hash_to_challenge(mu, w1)
            
            # 5. z = y + cs1
            cs1 = np.zeros_like(s1)
            for i in range(self.l):
                cs1[i] = (c[0] * s1[i]) % self.q  # 簡略
            z = y + cs1
            
            # 6. Reject sampling: z の係数が大きすぎたら再試行
            if np.abs(z).max() < gamma1 - self.beta:
                return (z, c, w1), attempt + 1
        
        return None, 100


# Dilithiumパラメータの比較
dilithium_params = {
    'Dilithium2 (ML-DSA-44)': {'k': 4, 'l': 4, 'eta': 2, 'サイズ(公開鍵)': '1312 B', 'サイズ(署名)': '2420 B', 'NISTレベル': 2},
    'Dilithium3 (ML-DSA-65)': {'k': 6, 'l': 5, 'eta': 4, 'サイズ(公開鍵)': '1952 B', 'サイズ(署名)': '3293 B', 'NISTレベル': 3},
    'Dilithium5 (ML-DSA-87)': {'k': 8, 'l': 7, 'eta': 2, 'サイズ(公開鍵)': '2592 B', 'サイズ(署名)': '4595 B', 'NISTレベル': 5},
}

print('CRYSTALS-Dilithiumパラメータセット:')
print(f'{"変種":<25} {"k":<4} {"l":<4} {"公開鍵":<12} {"署名":<12} {"NISTレベル"}')
print('-' * 65)
for name, p in dilithium_params.items():
    print(f'{name:<25} {p["k"]:<4} {p["l"]:<4} {p["サイズ(公開鍵)"]:<12} {p["サイズ(署名)"]:<12} Level {p["NISTレベル"]}')

# 従来の署名との比較
print()
print('従来署名との鍵・署名サイズ比較:')
comparison = {
    'RSA-2048': {'公開鍵': '256 B', '秘密鍵': '1232 B', '署名': '256 B', '量子耐性': '✗'},
    'ECDSA-256': {'公開鍵': '64 B', '秘密鍵': '32 B', '署名': '64 B', '量子耐性': '✗'},
    'Dilithium2': {'公開鍵': '1312 B', '秘密鍵': '2528 B', '署名': '2420 B', '量子耐性': '✓'},
    'Dilithium3': {'公開鍵': '1952 B', '秘密鍵': '4000 B', '署名': '3293 B', '量子耐性': '✓'},
}
print(f'{"方式":<15} {"公開鍵":<10} {"秘密鍵":<10} {"署名":<10} {"量子耐性"}')
print('-' * 55)
for name, p in comparison.items():
    print(f'{name:<15} {p["公開鍵"]:<10} {p["秘密鍵"]:<10} {p["署名"]:<10} {p["量子耐性"]}')

## 4. PQC移行戦略の可視化

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 鍵サイズ比較
schemes = ['RSA-2048', 'ECDSA-256', 'Kyber512\n(KEM)', 'Kyber768\n(KEM)', 'Dilithium2\n(署名)', 'Dilithium3\n(署名)']
pub_key_sizes = [256, 64, 800, 1184, 1312, 1952]
sig_sizes = [256, 64, 768, 1088, 2420, 3293]  # Kyberの場合は暗号文サイズ
colors = ['#e74c3c', '#e74c3c', '#2ecc71', '#27ae60', '#3498db', '#2980b9']

x = np.arange(len(schemes))
width = 0.35
bars1 = axes[0, 0].bar(x - width/2, pub_key_sizes, width, label='公開鍵サイズ', color=colors, alpha=0.8)
bars2 = axes[0, 0].bar(x + width/2, sig_sizes, width, label='署名/暗号文サイズ', color=colors, alpha=0.5, hatch='//')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(schemes, fontsize=9)
axes[0, 0].set_ylabel('バイト数', fontsize=11)
axes[0, 0].set_title('鍵・署名サイズ比較\n(赤=量子脆弱, 緑/青=PQC)', fontsize=12)
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3, axis='y')
axes[0, 0].set_yscale('log')

# 2. 安全レベル比較
categories = ['古典安全性\n(ビット)', '量子安全性\n(量子ビット)']
rsa2048 = [112, 0]   # RSA-2048: 古典112ビット, 量子0ビット
ecdsa256 = [128, 0]  # ECDSA-256: 古典128ビット, 量子0ビット
kyber512 = [178, 100]  # Kyber512: 高古典安全性, 量子100ビット
kyber768 = [230, 180]  # Kyber768

x = np.arange(len(categories))
width = 0.2
axes[0, 1].bar(x - 1.5*width, rsa2048, width, label='RSA-2048', color='#e74c3c', alpha=0.8)
axes[0, 1].bar(x - 0.5*width, ecdsa256, width, label='ECDSA-256', color='#e67e22', alpha=0.8)
axes[0, 1].bar(x + 0.5*width, kyber512, width, label='Kyber512', color='#2ecc71', alpha=0.8)
axes[0, 1].bar(x + 1.5*width, kyber768, width, label='Kyber768', color='#27ae60', alpha=0.8)
axes[0, 1].axhline(y=128, color='orange', linestyle='--', linewidth=1.5, label='128ビット安全基準')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(categories, fontsize=11)
axes[0, 1].set_ylabel('安全レベル (ビット)', fontsize=11)
axes[0, 1].set_title('古典 vs 量子安全性\n(量子0=量子攻撃に対して無防備)', fontsize=12)
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. PQC移行ロードマップ
timeline = [
    (2022, 2024, 'NIST PQC\n標準化完了', '#3498db'),
    (2024, 2027, 'ハイブリッド暗号\n(従来+PQC)', '#9b59b6'),
    (2027, 2030, '主要システムの\nPQC移行', '#2ecc71'),
    (2030, 2035, 'レガシー暗号\n廃止期限', '#e74c3c'),
]

for i, (start, end, label, color) in enumerate(timeline):
    axes[1, 0].barh(i, end - start, left=start, height=0.6, color=color, alpha=0.8)
    axes[1, 0].text((start + end) / 2, i, label, ha='center', va='center', fontsize=9, fontweight='bold', color='white')

axes[1, 0].axvline(x=2026, color='black', linestyle='--', linewidth=2, label='現在 (2026)')
axes[1, 0].set_xlim(2021, 2036)
axes[1, 0].set_yticks([])
axes[1, 0].set_xlabel('年', fontsize=11)
axes[1, 0].set_title('PQC移行ロードマップ', fontsize=12)
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3, axis='x')

# 4. ビジネスへの影響
sectors = ['金融\n決済', 'PKI\n証明書', 'TLS\n通信', 'IoT\nデバイス', 'クラウド\nストレージ', 'コード\n署名']
urgency = [9.5, 8.5, 8.0, 7.5, 7.0, 8.5]   # 緊急度 (10段階)
complexity = [8.0, 7.0, 6.5, 9.0, 6.0, 5.0]  # 移行複雑度

scatter = axes[1, 1].scatter(complexity, urgency, s=200, c=np.arange(len(sectors)), 
                               cmap='RdYlGn_r', alpha=0.8, zorder=5)
for i, sector in enumerate(sectors):
    axes[1, 1].annotate(sector, (complexity[i], urgency[i]), 
                         textcoords='offset points', xytext=(10, 5), fontsize=9)
axes[1, 1].axhline(y=8, color='red', linestyle='--', alpha=0.5, label='高緊急度ライン')
axes[1, 1].axvline(x=7, color='orange', linestyle='--', alpha=0.5, label='高複雑度ライン')
axes[1, 1].set_xlabel('移行複雑度 (高いほど困難)', fontsize=11)
axes[1, 1].set_ylabel('移行緊急度 (高いほど急務)', fontsize=11)
axes[1, 1].set_title('業種別PQC移行の優先度マトリクス', fontsize=12)
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xlim(4, 11)
axes[1, 1].set_ylim(6, 11)

plt.suptitle('耐量子暗号 (PQC) 総合分析', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('pqc_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('PQC総合分析グラフを保存しました')

## まとめ

| アルゴリズム | 用途 | 安全性根拠 | NIST FIPS | 量子耐性 |
|---|---|---|---|---|
| CRYSTALS-Kyber | 鍵交換(KEM) | Module-LWE | FIPS 203 | ✓ |
| CRYSTALS-Dilithium | デジタル署名 | Module-LWE+SIS | FIPS 204 | ✓ |
| SPHINCS+ | デジタル署名 | ハッシュ関数 | FIPS 205 | ✓ |
| FALCON | デジタル署名 | NTRU格子 | FIPS 206 | ✓ |

**次のノートブック**: `03_pqc_evaluation.ipynb` で実際のPQC評価・実装フレームワークを学びます。